# NB27 · Concurrency & Asyncio

`advanced/` · notebook 27 of 32 · 32 questions + a mini-project

Most programs spend their time *waiting*: for a server, a disk, a database.
Concurrency lets one program wait for many things at once. This notebook
covers the three tools Python gives you for that: threads (`threading`,
`queue`), thread pools (`concurrent.futures`), and the event loop
(`asyncio`).

**Two rules that make every answer here predictable.** Threads finish in
whatever order the operating system likes, so you always `join()` before
printing and sort results that arrive out of order. Async code in a
notebook uses top-level `await`, because a notebook cell already runs
inside an event loop. That's also why `asyncio.run()` (the right call in a
`.py` script) fails here: Q07 shows you.

**The GIL in one paragraph.** Standard CPython has a Global Interpreter Lock:
only one thread runs Python bytecode at a time. Threads still help with
waiting (I/O), because a thread that waits releases the lock. For heavy
*computation* you want processes, which have their own interpreter each.
Python 3.13 added an optional free-threaded build (`python3.13t`) with no
GIL, where threads really do run Python code in parallel. That makes locks
more important, not less.

**About `ProcessPoolExecutor`.** It has the same interface as
`ThreadPoolExecutor` but runs jobs in separate processes. To get a job to
another process, Python *pickles* the function and its arguments. Functions
defined in a notebook live in `__main__`, which worker processes often can't
import, so process pools are unreliable from Jupyter. Put the worker
function in a `.py` module and import it, or run the code as a script. The
questions here use threads and asyncio only; Q13 shows the pickling limit.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB22: generators
- [ ] NB24: context managers

## What you'll practise

- Concurrency vs parallelism; the GIL and the free-threaded build (3.13+)
- threading.Thread, join, race conditions, Lock; queue.Queue
- concurrent.futures: ThreadPoolExecutor map vs submit/as_completed
- ProcessPoolExecutor in concept (the notebook pickling caveat)
- asyncio: coroutines, await, top-level await in Jupyter, gather, create_task
- TaskGroup (3.11+), asyncio.timeout, Semaphore, to_thread
- Async iterators, async generators, async with

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB27-Q01 · L1 · Threads · Write

Run `backup("photos")` in a new thread. Wait for the thread to finish,
then print `finished`. Expected: `['photos']`.

**Starting code (already in the cell below):**

```python
import threading

finished = []

def backup(folder):
    finished.append(folder)
```

<details><summary><b>Hint</b></summary>

`threading.Thread(target=..., args=(...,))` creates the thread. `start()` runs it; `join()` waits for it.

</details>

<details><summary><b>Expected output</b></summary>

```text
['photos']
```

</details>

<details><summary><b>Solution</b></summary>

```python
worker = threading.Thread(target=backup, args=("photos",))
worker.start()
worker.join()
print(finished)
```

`target` is the function to run (without brackets: you pass the function, you don't call it). `args` is a tuple of arguments. `start()` launches the thread and returns at once; `join()` blocks until the thread is done. Printing before `join()` could show an empty list, depending on timing.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
worker = threading.Thread(target=backup, args=("photos",), name="backup-photos")
worker.start()
worker.join()
print(finished)
```

A `name` shows up in logs and debuggers, which matters once you have ten threads. Watch the comma in `args=("photos",)`: without it, `("photos")` is a plain string, and the thread would receive six arguments, one per letter.

</details>

In [ ]:
import threading

finished = []

def backup(folder):
    finished.append(folder)

# your code here


### NB27-Q02 · L1 · Coroutines · Write

Write a coroutine function `fetch_price(item)` that waits with
`await asyncio.sleep(0.01)` (pretending to call a server) and returns
`4.5`. Then `await` it at the top level of the cell and print:

```text
coffee costs 4.5
```

**Starting code (already in the cell below):**

```python
import asyncio
```

<details><summary><b>Hint</b></summary>

`async def` makes a coroutine function. Inside a notebook you can write `await` directly in the cell, outside any function.

</details>

<details><summary><b>Expected output</b></summary>

```text
coffee costs 4.5
```

</details>

<details><summary><b>Solution</b></summary>

```python
async def fetch_price(item):
    await asyncio.sleep(0.01)
    return 4.5

price = await fetch_price("coffee")
print("coffee costs", price)
```

`async def` defines a coroutine function. Calling it gives a coroutine object; `await` runs it until it returns. `await asyncio.sleep(...)` pauses this coroutine and lets the event loop run other work meanwhile. Jupyter allows `await` at the top level of a cell because the cell already runs inside an event loop.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
PRICES = {"coffee": 4.5, "tea": 3.0}

async def fetch_price(item):
    await asyncio.sleep(0.01)
    return PRICES[item]

price = await fetch_price("coffee")
print("coffee costs", price)
```

In a `.py` script there's no running loop, so you'd write `async def main(): ...` and start it once with `asyncio.run(main())` at the bottom. In a notebook, top-level `await` replaces that. The senior also makes the fake server answer by item, so the parameter isn't ignored.

</details>

In [ ]:
import asyncio

# your code here


### NB27-Q03 · L1 · Coroutines · Predict

A coroutine is called, but never awaited. Predict the output.

**What does this print?**

```python
async def send_receipt(email):
    print("sending receipt to", email)
    return True

result = send_receipt("ada@example.com")
print(type(result).__name__)
result.close()
print("done")
```

<details><summary><b>Hint</b></summary>

Calling an `async def` function doesn't run its body. What does it give back?

</details>

<details><summary><b>Expected output</b></summary>

```text
coroutine
done
```

</details>

<details><summary><b>Solution</b></summary>

Calling `send_receipt(...)` only *creates* a coroutine object. The body, including its `print`, never runs, so no receipt is sent. That's a silent bug: no error, no email. `result.close()` throws the unused coroutine away; without it Python prints a `RuntimeWarning: coroutine ... was never awaited`, which is the clue to look for.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
async def send_receipt(email):
    print("sending receipt to", email)
    return True

result = await send_receipt("ada@example.com")
print(result)
```

The fix is `await`. If you see "coroutine was never awaited" in your output, find the call that is missing its `await`. Linters such as Ruff and type checkers (NB26) flag unawaited coroutines before you run anything.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L2 · Easy

### NB27-Q04 · L2 · gather · Predict

Three health checks with different delays. `finished` records the order
they finish in. Predict both lines.

**What does this print?**

```python
import asyncio

finished = []

async def check(name, delay):
    await asyncio.sleep(delay)
    finished.append(name)
    return name.upper()

results = await asyncio.gather(check("db", 0.05), check("cache", 0.01), check("api", 0.03))
print(finished)
print(results)
```

<details><summary><b>Hint</b></summary>

All three run at the same time, so the shortest wait finishes first. But what order does `gather` put its *results* in?

</details>

<details><summary><b>Expected output</b></summary>

```text
['cache', 'api', 'db']
['DB', 'CACHE', 'API']
```

</details>

<details><summary><b>Solution</b></summary>

The checks run concurrently, so they finish shortest-first: `cache`, `api`, `db`. But `gather` returns results in the order you *passed* the coroutines, not the order they finished. That's what lets you line results up with their inputs.

</details>

<details><summary><b>Senior dev solution</b></summary>

Rely on `gather`'s ordering and `zip` the results with your inputs. If you want results as they arrive (to show progress), use `asyncio.as_completed`, and then the order is not fixed.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB27-Q05 · L2 · ThreadPoolExecutor · Write

`order_total` is slow because it calls a pricing service. Use a
`ThreadPoolExecutor` with 3 workers and `pool.map` to price all the
orders at once, then print the list of totals: `[7.0, 10.0, 5.0]`.

**Starting code (already in the cell below):**

```python
from concurrent.futures import ThreadPoolExecutor
import time

def order_total(order):
    time.sleep(0.01)          # pretend to call a slow pricing service
    return order["qty"] * order["price"]

orders = [
    {"id": 1, "qty": 2, "price": 3.5},
    {"id": 2, "qty": 1, "price": 10.0},
    {"id": 3, "qty": 4, "price": 1.25},
]
```

<details><summary><b>Hint</b></summary>

Open the pool with `with ThreadPoolExecutor(max_workers=3) as pool:`. `pool.map` works like `map`; wrap it in `list()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
[7.0, 10.0, 5.0]
```

</details>

<details><summary><b>Solution</b></summary>

```python
with ThreadPoolExecutor(max_workers=3) as pool:
    totals = list(pool.map(order_total, orders))
print(totals)
```

`pool.map` hands each order to a free worker thread and gives the results back **in input order**, however the threads happen to finish. The `with` block waits for every job and shuts the pool down. `time.sleep` releases the GIL, so the three waits overlap.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with ThreadPoolExecutor(max_workers=3) as pool:
    totals = list(pool.map(order_total, orders))
print(totals)
```

Same code: this is the idiomatic form. Seniors reach for `map` whenever every job is the same function and you want results in order. Always set `max_workers`; the default (based on CPU count) is a guess that may hammer a server you're calling.

</details>

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def order_total(order):
    time.sleep(0.01)          # pretend to call a slow pricing service
    return order["qty"] * order["price"]

orders = [
    {"id": 1, "qty": 2, "price": 3.5},
    {"id": 2, "qty": 1, "price": 10.0},
    {"id": 3, "qty": 4, "price": 1.25},
]

# your code here


### NB27-Q06 · L2 · queue.Queue · Write

Start `producer` in a thread. In the main thread, take lines out of
`inbox` with `get()` until you receive `None`, printing each one. Then
join the thread.

```text
got: GET /home 200
got: GET /cart 200
got: POST /pay 500
producer finished
```

**Starting code (already in the cell below):**

```python
import queue
import threading

lines = ["GET /home 200", "GET /cart 200", "POST /pay 500"]
inbox = queue.Queue()

def producer():
    for line in lines:
        inbox.put(line)
    inbox.put(None)        # sentinel: nothing more is coming
```

<details><summary><b>Hint</b></summary>

`inbox.get()` waits until an item is available. Loop with `while True:` and `break` on `None`.

</details>

<details><summary><b>Expected output</b></summary>

```text
got: GET /home 200
got: GET /cart 200
got: POST /pay 500
producer finished
```

</details>

<details><summary><b>Solution</b></summary>

```python
worker = threading.Thread(target=producer)
worker.start()
while True:
    line = inbox.get()
    if line is None:
        break
    print("got:", line)
worker.join()
print("producer finished")
```

`queue.Queue` is thread-safe: one thread can `put` while another `get`s, with no lock of your own. `get()` blocks until something arrives, so the consumer never spins or misses an item. The `None` *sentinel* tells the consumer to stop. Items come out in the order they went in (FIFO), so the output is predictable.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
worker = threading.Thread(target=producer)
worker.start()
for line in iter(inbox.get, None):
    print("got:", line)
worker.join()
print("producer finished")
```

`iter(callable, sentinel)` (NB22) calls `inbox.get()` repeatedly and stops when it returns `None`: the whole `while`/`break` loop in one line.

</details>

In [ ]:
import queue
import threading

lines = ["GET /home 200", "GET /cart 200", "POST /pay 500"]
inbox = queue.Queue()

def producer():
    for line in lines:
        inbox.put(line)
    inbox.put(None)        # sentinel: nothing more is coming

# your code here


### NB27-Q07 · L2 · Top-level await · Error

`run` is imported straight from `asyncio`, so `run(...)` here is
`asyncio.run(...)`. In a notebook every cell already runs inside an event
loop; the `await asyncio.sleep(0)` line makes this cell behave the same
way when run as a script. Which error does this raise?

**Which error does this raise?**

```python
import asyncio
from asyncio import run          # the same function as asyncio.run

async def daily_report():
    return "report ready"

await asyncio.sleep(0)           # we are inside a running event loop
print("starting report")
message = run(daily_report())
print(message)
```

<details><summary><b>Hint</b></summary>

`asyncio.run()` starts a brand-new event loop. Can it do that while one is already running in the same thread?

</details>

<details><summary><b>Expected output</b></summary>

```text
starting report
RuntimeError
```

</details>

<details><summary><b>Solution</b></summary>

`RuntimeError`: `asyncio.run()` creates a new event loop and refuses to start while another loop is running in the same thread. Jupyter is always running one, so `asyncio.run(main())`, copied from a script tutorial, fails in a notebook. Use `await main()` instead.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import asyncio

async def daily_report():
    return "report ready"

message = await daily_report()
print(message)
```

Rule of thumb: `asyncio.run()` belongs once, at the very bottom of a script (`if __name__ == "__main__": asyncio.run(main())`). Everywhere else, including notebooks, you `await`.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB27-Q08 · L2 · create_task · Write

Start two downloads as tasks with `asyncio.create_task`:
`download("report.pdf", 0.02)` and `download("logo.png", 0.01)`. While
they run, print `downloads started`. Then await the report task, then the
logo task, printing each result.

```text
downloads started
report.pdf downloaded
logo.png downloaded
```

**Starting code (already in the cell below):**

```python
import asyncio

async def download(file, delay):
    await asyncio.sleep(delay)
    return f"{file} downloaded"
```

<details><summary><b>Hint</b></summary>

`asyncio.create_task(coro)` schedules the coroutine and returns a task right away. `await task` gives you its return value.

</details>

<details><summary><b>Expected output</b></summary>

```text
downloads started
report.pdf downloaded
logo.png downloaded
```

</details>

<details><summary><b>Solution</b></summary>

```python
report = asyncio.create_task(download("report.pdf", 0.02))
logo = asyncio.create_task(download("logo.png", 0.01))
print("downloads started")
print(await report)
print(await logo)
```

`create_task` schedules a coroutine to run as soon as the current code pauses, and hands you a `Task`. The `print` runs first because nothing has paused yet. At `await report` both downloads run side by side, so the total wait is about 0.02 s, not 0.03 s. By the time the report is done, the logo is already finished, and `await logo` returns at once.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
report = asyncio.create_task(download("report.pdf", 0.02))
logo = asyncio.create_task(download("logo.png", 0.01))
print("downloads started")
for message in await asyncio.gather(report, logo):
    print(message)
```

`gather` accepts tasks as well as coroutines and keeps argument order. Keep a reference to every task you create: the event loop only holds a weak reference, so a task nobody points to can be garbage collected mid-run.

</details>

In [ ]:
import asyncio

async def download(file, delay):
    await asyncio.sleep(delay)
    return f"{file} downloaded"

# your code here


### NB27-Q09 · L2 · Concurrency vs parallelism · Write

For each job, print the right tool. I/O-bound jobs (mostly waiting) get
`threads or asyncio`; CPU-bound jobs (mostly computing) get `processes`.

```text
resize 5,000 photos -> processes
download 200 web pages -> threads or asyncio
query a remote database -> threads or asyncio
train a small model -> processes
```

**Starting code (already in the cell below):**

```python
jobs = [
    ("resize 5,000 photos", "cpu"),
    ("download 200 web pages", "io"),
    ("query a remote database", "io"),
    ("train a small model", "cpu"),
]
```

<details><summary><b>Hint</b></summary>

Loop over the pairs and choose the text with `if`.

</details>

<details><summary><b>Expected output</b></summary>

```text
resize 5,000 photos -> processes
download 200 web pages -> threads or asyncio
query a remote database -> threads or asyncio
train a small model -> processes
```

</details>

<details><summary><b>Solution</b></summary>

```python
for job, kind in jobs:
    if kind == "io":
        tool = "threads or asyncio"
    else:
        tool = "processes"
    print(job, "->", tool)
```

*Concurrency* means juggling several tasks that overlap in time; *parallelism* means running them at literally the same moment on different cores. Waiting doesn't need a core, so threads or asyncio are enough for I/O: while one task waits, another runs. Computing does need cores, and the GIL lets only one thread run Python code at a time, so CPU-heavy work needs separate processes.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
TOOL_FOR = {"io": "threads or asyncio", "cpu": "processes"}

for job, kind in jobs:
    print(job, "->", TOOL_FOR[kind])
```

A lookup table replaces the `if`, and an unknown kind raises `KeyError` instead of silently becoming `processes`. On the free-threaded build (3.13+, check with `sys._is_gil_enabled()`), threads can run CPU-bound Python in parallel too; libraries like NumPy already release the GIL inside their heavy loops.

</details>

In [ ]:
jobs = [
    ("resize 5,000 photos", "cpu"),
    ("download 200 web pages", "io"),
    ("query a remote database", "io"),
    ("train a small model", "cpu"),
]

# your code here


---

## L3 · Intermediate

### NB27-Q10 · L3 · Lock · Write

Four threads each record 10,000 page visits by adding 1 to `visits`. Write
`record_visits(n)` so the update happens inside `with lock:`, start four
threads running `record_visits(10_000)`, join them all, and print
`visits`. It must be `40000` every time.

**Starting code (already in the cell below):**

```python
import threading

visits = 0
lock = threading.Lock()
```

<details><summary><b>Hint</b></summary>

`visits` is a global you change, so the function needs `global visits`. Keep the threads in a list so you can `start()` all, then `join()` all.

</details>

<details><summary><b>Expected output</b></summary>

```text
40000
```

</details>

<details><summary><b>Solution</b></summary>

```python
def record_visits(n):
    global visits
    for _ in range(n):
        with lock:
            visits += 1

threads = [threading.Thread(target=record_visits, args=(10_000,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(visits)
```

`visits += 1` is three steps: read `visits`, add 1, write it back. If another thread runs between the read and the write, one update is lost (Q11 shows exactly how). `with lock:` lets only one thread at a time through those three steps, so no update can be lost. Starting all threads before joining any lets them run together.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def record_visits(n):
    global visits
    local_count = 0
    for _ in range(n):
        local_count += 1
    with lock:
        visits += local_count

threads = [threading.Thread(target=record_visits, args=(10_000,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(visits)
```

Taking a lock 40,000 times is slow. Seniors keep shared state out of the hot loop: each thread counts privately and takes the lock once. On a normal GIL build the unlocked version often *happens* to print 40000, which is exactly why races slip through testing. On the free-threaded build it loses updates quickly.

</details>

In [ ]:
import threading

visits = 0
lock = threading.Lock()

# your code here


### NB27-Q11 · L3 · Race conditions · Predict

A race condition, replayed in slow motion. Two threads each deposit 50
into an account holding 100. The `Barrier` makes both threads finish their
*read* before either *writes*, which is what unlucky scheduling does now
and then. Predict the output.

**What does this print?**

```python
import threading

balance = 100
both_have_read = threading.Barrier(2)

def deposit(amount):
    global balance
    current = balance            # 1. read
    both_have_read.wait()        # wait here until the other thread has read too
    balance = current + amount   # 2. write

threads = [threading.Thread(target=deposit, args=(50,)) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(balance)
```

<details><summary><b>Hint</b></summary>

What value of `balance` did each thread read? What does each one write?

</details>

<details><summary><b>Expected output</b></summary>

```text
150
```

</details>

<details><summary><b>Solution</b></summary>

Both threads read `100` before either writes. Each then writes `100 + 50 = 150`. The second write overwrites the first, so one deposit is lost: `150`, not `200`. That's a *lost update*. Without the barrier this only happens when the scheduler switches threads at the wrong moment, maybe once in a million runs, which is what makes races so hard to debug.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import threading

balance = 100
lock = threading.Lock()

def deposit(amount):
    global balance
    with lock:
        current = balance
        balance = current + amount

threads = [threading.Thread(target=deposit, args=(50,)) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(balance)
```

The fix puts the read *and* the write inside one `with lock:`, so the pair can't be split. This version prints `200` every time. A `Barrier` is a handy tool for writing tests that force a bad interleaving on purpose.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB27-Q12 · L3 · Blocking the event loop · Fix

The heartbeat should keep beating while the report waits for its slow
server, so the log should be:

```text
['beat 0', 'beat 1', 'beat 2', 'report done']
```

Instead the report blocks everything. Fix it.

**Buggy code (copied into the cell below):**

```python
import asyncio
import time

log = []

async def heartbeat():
    for beat in range(3):
        log.append(f"beat {beat}")
        await asyncio.sleep(0)

async def nightly_report():
    time.sleep(0.05)            # pretend to wait for a slow server
    log.append("report done")

await asyncio.gather(nightly_report(), heartbeat())
print(log)
```

<details><summary><b>Hint</b></summary>

The event loop can only switch tasks at an `await`. Does `time.sleep` ever give control back?

</details>

<details><summary><b>Expected output</b></summary>

```text
['beat 0', 'beat 1', 'beat 2', 'report done']
```

</details>

<details><summary><b>Solution</b></summary>

```python
import asyncio

log = []

async def heartbeat():
    for beat in range(3):
        log.append(f"beat {beat}")
        await asyncio.sleep(0)

async def nightly_report():
    await asyncio.sleep(0.05)   # pretend to wait for a slow server
    log.append("report done")

await asyncio.gather(nightly_report(), heartbeat())
print(log)
```

asyncio runs every task on **one** thread and switches only at `await`. `time.sleep(0.05)` freezes that thread, so the heartbeat can't run until the report is done. `await asyncio.sleep(0.05)` pauses only this coroutine and hands control back to the loop, which runs the heartbeat. `await asyncio.sleep(0)` means "let others run, then continue".

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import asyncio

log = []

async def heartbeat():
    for beat in range(3):
        log.append(f"beat {beat}")
        await asyncio.sleep(0)

async def nightly_report():
    await asyncio.sleep(0.05)
    log.append("report done")

async with asyncio.TaskGroup() as tg:
    tg.create_task(nightly_report())
    tg.create_task(heartbeat())
print(log)
```

The same trap hides in `requests.get`, file reads and heavy loops inside `async def`. If you must call blocking code, push it to a thread with `asyncio.to_thread` (Q22). Running with `asyncio` debug mode on logs any step that blocks the loop for too long.

</details>

In [ ]:
import asyncio
import time

log = []

async def heartbeat():
    for beat in range(3):
        log.append(f"beat {beat}")
        await asyncio.sleep(0)

async def nightly_report():
    time.sleep(0.05)            # pretend to wait for a slow server
    log.append("report done")

await asyncio.gather(nightly_report(), heartbeat())
print(log)


### NB27-Q13 · L3 · ProcessPoolExecutor · Error

`ProcessPoolExecutor` sends every job to another process by **pickling**
the function and its arguments. Here's what happens to a function it
can't find by name. Which error does this raise?

**Which error does this raise?**

```python
import pickle

double = lambda price: price * 2
print("pickling the job...")
data = pickle.dumps(double)
```

<details><summary><b>Hint</b></summary>

`pickle` stores a function as its module and name, and the worker looks it up again. What is a lambda's name?

</details>

<details><summary><b>Expected output</b></summary>

```text
pickling the job...
PicklingError
```

</details>

<details><summary><b>Solution</b></summary>

`PicklingError`: pickle saves a function as a *reference* (module + name) and the worker process imports it by that name. A lambda is called `<lambda>`, which can't be looked up, so it can't be sent. The notebook version of the problem: functions defined in a notebook belong to `__main__`, which a fresh worker process can't import. That's why process pools want their worker functions in a real `.py` module.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import pickle

def double(price):
    return price * 2

data = pickle.dumps(double)
print(pickle.loads(data)(21))     # safe: we made this data ourselves
```

A named, top-level function pickles fine *within* one process. For a process pool, seniors put it in a module (say `workers.py`), import it, and run the pool under `if __name__ == "__main__":`, because on Windows and macOS each worker re-imports the main module.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB27-Q14 · L3 · as_completed · Write

Check stock for every item with `pool.submit`, one job per item, using
3 workers. Collect the results with `as_completed` into a dict, then print
the items in alphabetical order.

```text
apples: 12
bread: 0
milk: 7
```

**Starting code (already in the cell below):**

```python
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

STOCK = {"apples": 12, "bread": 0, "milk": 7}

def check_stock(item):
    time.sleep(0.01)
    return item, STOCK[item]

items = ["milk", "apples", "bread"]
```

<details><summary><b>Hint</b></summary>

`pool.submit(func, arg)` returns a future. `as_completed(futures)` yields each future as it finishes; `future.result()` gives its return value.

</details>

<details><summary><b>Expected output</b></summary>

```text
apples: 12
bread: 0
milk: 7
```

</details>

<details><summary><b>Solution</b></summary>

```python
levels = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(check_stock, item) for item in items]
    for future in as_completed(futures):
        item, count = future.result()
        levels[item] = count
for item in sorted(levels):
    print(f"{item}: {levels[item]}")
```

`submit` starts one job and returns a `Future`, a placeholder for a result that will exist later. `as_completed` hands futures back in **finishing order**, which changes from run to run. That's why you collect into a dict and sort before printing. Compare `map` (Q05), which always keeps input order.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with ThreadPoolExecutor(max_workers=3) as pool:
    future_to_item = {pool.submit(check_stock, item): item for item in items}
    levels = dict(future.result() for future in as_completed(future_to_item))
for item, count in sorted(levels.items()):
    print(f"{item}: {count}")
```

The `{future: item}` dict is the standard pattern from the `concurrent.futures` docs: if a job fails, you still know *which* item it was for. Use `submit`/`as_completed` when you want to react to each result as soon as it lands (progress bars, early exit); use `map` otherwise.

</details>

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

STOCK = {"apples": 12, "bread": 0, "milk": 7}

def check_stock(item):
    time.sleep(0.01)
    return item, STOCK[item]

items = ["milk", "apples", "bread"]

# your code here


### NB27-Q15 · L3 · asyncio.timeout · Write

Allow each weather fetch at most 0.02 seconds, using
`async with asyncio.timeout(0.02):`. Fetch Oslo (delay `0`) and then Lima
(delay `0.05`). Print the result, or `<city>: timed out` when the time
runs out.

```text
Oslo: sunny
Lima: timed out
```

**Starting code (already in the cell below):**

```python
import asyncio

async def fetch_weather(city, delay):
    await asyncio.sleep(delay)
    return f"{city}: sunny"

requests = [("Oslo", 0), ("Lima", 0.05)]
```

<details><summary><b>Hint</b></summary>

When the time runs out, the `async with` block raises `TimeoutError`. Put the `async with` inside a `try`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Oslo: sunny
Lima: timed out
```

</details>

<details><summary><b>Solution</b></summary>

```python
for city, delay in requests:
    try:
        async with asyncio.timeout(0.02):
            result = await fetch_weather(city, delay)
        print(result)
    except TimeoutError:
        print(f"{city}: timed out")
```

`asyncio.timeout(seconds)` (3.11+) cancels the code inside it when the deadline passes, and turns that cancellation into a `TimeoutError` as it leaves the block. Lima's 0.05 s wait is cut off at 0.02 s, so the program never waits the full time.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
async def weather_or_timeout(city, delay, limit=0.02):
    try:
        async with asyncio.timeout(limit):
            return await fetch_weather(city, delay)
    except TimeoutError:
        return f"{city}: timed out"

for result in await asyncio.gather(*(weather_or_timeout(c, d) for c, d in requests)):
    print(result)
```

Wrapping the timeout in a helper keeps the policy in one place, and `gather` runs both fetches at once. Before 3.11 you'd use `asyncio.wait_for(coro, timeout)`, which still works. Every network call in production code needs *some* timeout; the default is to wait forever.

</details>

In [ ]:
import asyncio

async def fetch_weather(city, delay):
    await asyncio.sleep(delay)
    return f"{city}: sunny"

requests = [("Oslo", 0), ("Lima", 0.05)]

# your code here


### NB27-Q16 · L3 · TaskGroup · Write

Fetch profiles 3, 1 and 2 at the same time using
`async with asyncio.TaskGroup() as tg:` and `tg.create_task(...)`. After
the block, print each task's result in the order you created them.

```text
user 3: Linus
user 1: Ada
user 2: Grace
```

**Starting code (already in the cell below):**

```python
import asyncio

NAMES = {1: "Ada", 2: "Grace", 3: "Linus"}

async def fetch_profile(user_id):
    await asyncio.sleep(0.01)
    return f"user {user_id}: {NAMES[user_id]}"
```

<details><summary><b>Hint</b></summary>

Keep the tasks in a list. When the `async with` block ends, every task is done and `task.result()` gives its value.

</details>

<details><summary><b>Expected output</b></summary>

```text
user 3: Linus
user 1: Ada
user 2: Grace
```

</details>

<details><summary><b>Solution</b></summary>

```python
async with asyncio.TaskGroup() as tg:
    tasks = [tg.create_task(fetch_profile(user_id)) for user_id in [3, 1, 2]]
for task in tasks:
    print(task.result())
```

A `TaskGroup` (3.11+) waits for all its tasks when the `async with` block ends, so after the block every result is ready. If any task fails, the group cancels the others and raises the errors together (Q27). That's *structured concurrency*: no task can outlive the block that started it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
async with asyncio.TaskGroup() as tg:
    tasks = [tg.create_task(fetch_profile(user_id)) for user_id in (3, 1, 2)]
print(*(task.result() for task in tasks), sep="\n")
```

Prefer `TaskGroup` over loose `create_task` calls: forgotten tasks and swallowed exceptions become impossible. Use `gather` when you want a plain list of results back, or `return_exceptions=True` (Q29).

</details>

In [ ]:
import asyncio

NAMES = {1: "Ada", 2: "Grace", 3: "Linus"}

async def fetch_profile(user_id):
    await asyncio.sleep(0.01)
    return f"user {user_id}: {NAMES[user_id]}"

# your code here


### NB27-Q17 · L3 · Async generators · Write

`read_sensor` is an async generator. Use `async for` to go through
`read_sensor([21.5, 22.0, 35.2, 22.4])` and print each reading, adding
` (alert)` when it is above `30`.

```text
21.5
22.0
35.2 (alert)
22.4
```

**Starting code (already in the cell below):**

```python
import asyncio

async def read_sensor(readings):
    for value in readings:
        await asyncio.sleep(0)       # pretend to wait for the next reading
        yield value
```

<details><summary><b>Hint</b></summary>

A normal `for` can't loop over an async generator. `async for` can, and it works at the top level of a notebook cell.

</details>

<details><summary><b>Expected output</b></summary>

```text
21.5
22.0
35.2 (alert)
22.4
```

</details>

<details><summary><b>Solution</b></summary>

```python
async for value in read_sensor([21.5, 22.0, 35.2, 22.4]):
    if value > 30:
        print(value, "(alert)")
    else:
        print(value)
```

An `async def` with `yield` is an *async generator*: like the generators from NB22, but it may `await` between values. `async for` asks for each value with `await`, so while it waits for the next reading the event loop can run other tasks.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
readings = [value async for value in read_sensor([21.5, 22.0, 35.2, 22.4])]
for value in readings:
    print(f"{value}{' (alert)' if value > 30 else ''}")
```

`[x async for x in agen]` is an async comprehension. Here it collects first and prints after, which suits small streams. For an endless stream (a real sensor) keep the `async for` loop so you react to each value as it arrives.

</details>

In [ ]:
import asyncio

async def read_sensor(readings):
    for value in readings:
        await asyncio.sleep(0)       # pretend to wait for the next reading
        yield value

# your code here


### NB27-Q18 · L3 · create_task · Predict

Tasks only start running when the current code pauses at an `await`.
Predict the log.

**What does this print?**

```python
import asyncio

log = []

async def job(name):
    log.append(f"{name} start")
    await asyncio.sleep(0)
    log.append(f"{name} end")

async def main():
    first = asyncio.create_task(job("A"))
    second = asyncio.create_task(job("B"))
    log.append("main before await")
    await first
    await second
    log.append("main done")

await main()
print(*log, sep="\n")
```

<details><summary><b>Hint</b></summary>

`create_task` schedules a task but doesn't run it yet. When `main` pauses, tasks run in the order they were created, each until its next `await`.

</details>

<details><summary><b>Expected output</b></summary>

```text
main before await
A start
B start
A end
B end
main done
```

</details>

<details><summary><b>Solution</b></summary>

`create_task` only *schedules*, so `main` logs first. At `await first`, main pauses and the loop runs A until its `await` ("A start"), then B until its `await` ("B start"). Next round: A finishes, then B finishes. Finally main resumes. One thread, taking turns at every `await`: that's cooperative multitasking.

</details>

<details><summary><b>Senior dev solution</b></summary>

This turn-taking is why asyncio code rarely needs locks: nothing else can run between two lines unless there's an `await` between them. Put an `await` inside a read-modify-write, though, and you can race again; `asyncio.Lock` exists for that case.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB27-Q19 · L3 · ThreadPoolExecutor · Refactor

This counts words on several pages with hand-made threads and a
pre-sized results list. Rewrite it with `ThreadPoolExecutor` and `map`.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
import threading

def word_count(text):
    return len(text.split())

pages = ["the cat sat", "a quick brown fox jumps", "hello world"]
counts = [None] * len(pages)

def worker(index):
    counts[index] = word_count(pages[index])

threads = []
for i in range(len(pages)):
    t = threading.Thread(target=worker, args=(i,))
    threads.append(t)
    t.start()
for t in threads:
    t.join()
print(counts)
```

<details><summary><b>Hint</b></summary>

`pool.map(word_count, pages)` returns the results in order. You don't need `worker`, the index, or the pre-sized list.

</details>

<details><summary><b>Expected output</b></summary>

```text
[3, 5, 2]
```

</details>

<details><summary><b>Solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor

def word_count(text):
    return len(text.split())

pages = ["the cat sat", "a quick brown fox jumps", "hello world"]
with ThreadPoolExecutor(max_workers=3) as pool:
    counts = list(pool.map(word_count, pages))
print(counts)
```

The pool does all the bookkeeping: it starts the threads, matches each result to its input, and waits for everything at the end of the `with` block. No shared list, no indexes, no join loop.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor

def word_count(text):
    return len(text.split())

pages = ["the cat sat", "a quick brown fox jumps", "hello world"]
with ThreadPoolExecutor(max_workers=min(8, len(pages))) as pool:
    counts = list(pool.map(word_count, pages))
print(counts)
```

Also, raw threads swallow exceptions (they print a traceback and the result stays `None`), while `pool.map` re-raises a worker's exception in your code. The senior caps the pool size so a list of 10,000 pages doesn't become 10,000 threads. (`word_count` is pure computation, so in real code threads won't speed it up; this is about the shape of the code.)

</details>

In [ ]:
import threading

def word_count(text):
    return len(text.split())

pages = ["the cat sat", "a quick brown fox jumps", "hello world"]
counts = [None] * len(pages)

def worker(index):
    counts[index] = word_count(pages[index])

threads = []
for i in range(len(pages)):
    t = threading.Thread(target=worker, args=(i,))
    threads.append(t)
    t.start()
for t in threads:
    t.join()
print(counts)


---

## L4 · Somewhat Difficult

### NB27-Q20 · L4 · asyncio.Queue · Write

Two baristas share one order queue. Write:

- `producer(queue)`: puts every order from `orders` on the queue, then
  one `None` per barista;
- `barista(queue)`: takes orders until it gets `None`, and for each one
  waits `await asyncio.sleep(0.01)` and appends `"made <order>"` to `made`.

Create an `asyncio.Queue`, run the producer and two baristas together
with `gather`, then print `sorted(made)` and how many were made.

```text
['made espresso', 'made latte', 'made mocha', 'made tea']
4 drinks made
```

**Starting code (already in the cell below):**

```python
import asyncio

orders = ["latte", "tea", "mocha", "espresso"]
made = []
```

<details><summary><b>Hint</b></summary>

`await queue.put(item)` and `await queue.get()`. Two baristas need two `None` sentinels, or one of them waits forever.

</details>

<details><summary><b>Expected output</b></summary>

```text
['made espresso', 'made latte', 'made mocha', 'made tea']
4 drinks made
```

</details>

<details><summary><b>Solution</b></summary>

```python
async def producer(queue):
    for order in orders:
        await queue.put(order)
    await queue.put(None)
    await queue.put(None)

async def barista(queue):
    while True:
        order = await queue.get()
        if order is None:
            break
        await asyncio.sleep(0.01)
        made.append(f"made {order}")

queue = asyncio.Queue()
await asyncio.gather(producer(queue), barista(queue), barista(queue))
print(sorted(made))
print(len(made), "drinks made")
```

`asyncio.Queue` is the async version of `queue.Queue`: `get()` is awaited, so a barista with nothing to do pauses without blocking the others. Each barista stops at its own `None`, so you need one per consumer. Which barista makes which drink depends on timing, so the result is sorted before printing.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
async def barista(queue):
    while True:
        order = await queue.get()
        await asyncio.sleep(0.01)
        made.append(f"made {order}")
        queue.task_done()

queue = asyncio.Queue()
for order in orders:
    queue.put_nowait(order)
async with asyncio.TaskGroup() as tg:
    workers = [tg.create_task(barista(queue)) for _ in range(2)]
    await queue.join()
    for worker in workers:
        worker.cancel()
print(sorted(made))
print(len(made), "drinks made")
```

The sentinel-free pattern: each worker calls `task_done()`, `queue.join()` waits until every item is processed, then the idle workers are cancelled. It works for any number of workers, and nothing breaks if someone adds a third barista and forgets a `None`. Use `asyncio.Queue(maxsize=...)` to slow a fast producer down.

</details>

In [ ]:
import asyncio

orders = ["latte", "tea", "mocha", "espresso"]
made = []

# your code here


### NB27-Q21 · L4 · Semaphore · Write

Write `download(page, limit)` that, inside `async with limit:`, adds 1 to
`active`, updates `peak` with the highest `active` seen so far, waits
`await asyncio.sleep(0.01)`, subtracts 1 from `active` and returns
`"page <n> ok"`. Then create `asyncio.Semaphore(2)`, download pages 1 to 5
with `gather`, and print:

```text
['page 1 ok', 'page 2 ok', 'page 3 ok', 'page 4 ok', 'page 5 ok']
peak downloads at once: 2
```

**Starting code (already in the cell below):**

```python
import asyncio

active = 0
peak = 0
```

<details><summary><b>Hint</b></summary>

A semaphore is a counter of free slots: `async with limit:` takes a slot and waits if none are free. `active` and `peak` are globals you change.

</details>

<details><summary><b>Expected output</b></summary>

```text
['page 1 ok', 'page 2 ok', 'page 3 ok', 'page 4 ok', 'page 5 ok']
peak downloads at once: 2
```

</details>

<details><summary><b>Solution</b></summary>

```python
async def download(page, limit):
    global active, peak
    async with limit:
        active += 1
        peak = max(peak, active)
        await asyncio.sleep(0.01)
        active -= 1
        return f"page {page} ok"

limit = asyncio.Semaphore(2)
results = await asyncio.gather(*(download(page, limit) for page in range(1, 6)))
print(results)
print("peak downloads at once:", peak)
```

`Semaphore(2)` has two slots. The first two downloads take them; the other three wait at `async with limit:` until a slot frees up. So at most two requests are ever in flight, which is how you stay polite to a server with a rate limit. `gather` still returns results in page order.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
async def download(page, limit):
    global active, peak
    async with limit:
        active += 1
        peak = max(peak, active)
        try:
            await asyncio.sleep(0.01)
            return f"page {page} ok"
        finally:
            active -= 1

limit = asyncio.Semaphore(2)
results = await asyncio.gather(*(download(page, limit) for page in range(1, 6)))
print(results)
print("peak downloads at once:", peak)
```

The `try`/`finally` keeps `active` correct even if the download raises or is cancelled by a timeout. The `async with` already guarantees the semaphore slot is released; do the same for your own counters. No lock is needed because there's no `await` between reading and writing `active`.

</details>

In [ ]:
import asyncio

active = 0
peak = 0

# your code here


### NB27-Q22 · L4 · to_thread · Write

`blocking_backup` is an old, blocking function with no async version. Run
it with `asyncio.to_thread` alongside `heartbeat()` using `gather`, then
print `log` one entry per line.

```text
beat 0
beat 1
beat 2
backup finished
```

The backup only finishes after the heartbeat sets the event, so this
output proves the loop stayed free while the backup blocked.

**Starting code (already in the cell below):**

```python
import asyncio
import threading

backup_may_finish = threading.Event()
log = []

def blocking_backup():
    backup_may_finish.wait(timeout=1)     # blocks this thread until the event is set
    log.append("backup finished")

async def heartbeat():
    for beat in range(3):
        log.append(f"beat {beat}")
        await asyncio.sleep(0)
    backup_may_finish.set()
```

<details><summary><b>Hint</b></summary>

`asyncio.to_thread(func)` returns an awaitable, so it can go straight into `gather` next to `heartbeat()`. Pass the function itself, not `func()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
beat 0
beat 1
beat 2
backup finished
```

</details>

<details><summary><b>Solution</b></summary>

```python
await asyncio.gather(asyncio.to_thread(blocking_backup), heartbeat())
for entry in log:
    print(entry)
```

`asyncio.to_thread` runs a blocking function in a worker thread and gives you something to `await`. The backup blocks *its own* thread, so the event loop keeps running the heartbeat. Calling `blocking_backup()` directly in a coroutine would freeze the loop: no beats, and a one-second hang until the event's timeout.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
await asyncio.gather(asyncio.to_thread(blocking_backup), heartbeat())
print(*log, sep="\n")
```

`to_thread` is the bridge for libraries without async support (old database drivers, `requests`, file compression). For CPU-heavy work it won't help on a GIL build; use `loop.run_in_executor` with a process pool from a script. The `Event` here makes the order certain; real code just awaits the result.

</details>

In [ ]:
import asyncio
import threading

backup_may_finish = threading.Event()
log = []

def blocking_backup():
    backup_may_finish.wait(timeout=1)     # blocks this thread until the event is set
    log.append("backup finished")

async def heartbeat():
    for beat in range(3):
        log.append(f"beat {beat}")
        await asyncio.sleep(0)
    backup_may_finish.set()

# your code here


### NB27-Q23 · L4 · Race conditions · Fix

Two cash machines withdraw 80 from an account holding 100 at the same
moment. The barrier makes both threads pass the balance check before
either withdraws (what bad timing does in real life), and the account
goes negative.

Fix it with a `threading.Lock` so the check and the withdrawal happen as
one step, and delete the barrier (its only job was to replay the bad
timing). Expected:

```text
['approved', 'declined']
balance: 20
```

**Buggy code (copied into the cell below):**

```python
import threading

balance = 100
results = []
both_checked = threading.Barrier(2)

def withdraw(amount):
    global balance
    if balance >= amount:
        both_checked.wait()
        balance -= amount
        results.append("approved")
    else:
        results.append("declined")

machines = [threading.Thread(target=withdraw, args=(80,)) for _ in range(2)]
for m in machines:
    m.start()
for m in machines:
    m.join()
print(sorted(results))
print("balance:", balance)
```

<details><summary><b>Hint</b></summary>

Create a lock next to `balance`, and put the whole `if`/`else` inside `with lock:`.

</details>

<details><summary><b>Expected output</b></summary>

```text
['approved', 'declined']
balance: 20
```

</details>

<details><summary><b>Solution</b></summary>

```python
import threading

balance = 100
results = []
lock = threading.Lock()

def withdraw(amount):
    global balance
    with lock:
        if balance >= amount:
            balance -= amount
            results.append("approved")
        else:
            results.append("declined")

machines = [threading.Thread(target=withdraw, args=(80,)) for _ in range(2)]
for m in machines:
    m.start()
for m in machines:
    m.join()
print(sorted(results))
print("balance:", balance)
```

This race is *check-then-act*: the check (`balance >= amount`) and the action (`balance -= amount`) are separate steps, and another thread can slip in between. Holding the lock across both makes them one atomic step: the second machine sees a balance of 20 and declines. Which machine wins varies, so the results are sorted.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import threading

class Account:
    def __init__(self, balance):
        self.balance = balance
        self._lock = threading.Lock()

    def withdraw(self, amount):
        with self._lock:
            if self.balance < amount:
                return "declined"
            self.balance -= amount
            return "approved"

account = Account(100)
results = []
machines = [threading.Thread(target=lambda: results.append(account.withdraw(80)))
            for _ in range(2)]
for m in machines:
    m.start()
for m in machines:
    m.join()
print(sorted(results))
print("balance:", account.balance)
```

Seniors keep the lock *inside* the object that owns the data, so no caller can forget it. Keeping the barrier and adding a lock around it would deadlock: one thread holds the lock while waiting at the barrier for a thread that's waiting for the lock. `list.append` is thread-safe in CPython, so `results` needs no lock of its own.

</details>

In [ ]:
import threading

balance = 100
results = []
both_checked = threading.Barrier(2)

def withdraw(amount):
    global balance
    if balance >= amount:
        both_checked.wait()
        balance -= amount
        results.append("approved")
    else:
        results.append("declined")

machines = [threading.Thread(target=withdraw, args=(80,)) for _ in range(2)]
for m in machines:
    m.start()
for m in machines:
    m.join()
print(sorted(results))
print("balance:", balance)


### NB27-Q24 · L4 · async with · Predict

An async context manager around two print jobs. The second job is too big
and raises. Predict the log.

**What does this print?**

```python
import asyncio

log = []

class PrinterSession:
    async def __aenter__(self):
        await asyncio.sleep(0)
        log.append("connect")
        return self

    async def __aexit__(self, exc_type, exc, tb):
        await asyncio.sleep(0)
        log.append(f"disconnect ({exc_type.__name__ if exc_type else 'clean'})")
        return False

async def print_job(pages):
    async with PrinterSession():
        log.append("printing")
        if pages > 100:
            raise ValueError("too many pages")
        log.append("printed")

for pages in (5, 500):
    try:
        await print_job(pages)
    except ValueError:
        log.append("job rejected")
print(*log, sep="\n")
```

<details><summary><b>Hint</b></summary>

`__aexit__` runs whether or not the block raised, and it receives the exception type. Returning `False` lets the exception carry on.

</details>

<details><summary><b>Expected output</b></summary>

```text
connect
printing
printed
disconnect (clean)
connect
printing
disconnect (ValueError)
job rejected
```

</details>

<details><summary><b>Solution</b></summary>

`async with` works like `with` (NB24) but awaits `__aenter__` and `__aexit__`, so connecting and disconnecting can do I/O. The first job runs cleanly. The second raises inside the block: `__aexit__` still runs and sees `ValueError`, returns `False`, so the error continues to the `except` in the loop, which logs "job rejected" last.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import asyncio
from contextlib import asynccontextmanager

log = []

@asynccontextmanager
async def printer_session():
    await asyncio.sleep(0)
    log.append("connect")
    try:
        yield
    finally:
        await asyncio.sleep(0)
        log.append("disconnect")

async with printer_session():
    log.append("printing")
print(*log, sep="\n")
```

`@asynccontextmanager` (NB24) writes the same thing as a generator: setup before `yield`, cleanup in `finally`. Most real async resources (database pools, HTTP sessions) are used this way.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB27-Q25 · L4 · Async iterators · Write

Write a class `UserPages` that works with `async for`: `__aiter__`
returns `self`, and `__anext__` fetches the next page with `fetch_page`
(starting at page 1) and returns its list of users. When a page comes
back empty, raise `StopAsyncIteration`. Then loop over `UserPages()`:

```text
page 1: ['ada', 'grace']
page 2: ['linus']
```

**Starting code (already in the cell below):**

```python
import asyncio

PAGES = {1: ["ada", "grace"], 2: ["linus"], 3: []}   # an empty page means: no more users

async def fetch_page(number):
    await asyncio.sleep(0)
    return PAGES[number]
```

<details><summary><b>Hint</b></summary>

Store the current page number on `self` in `__init__`. `__anext__` is an `async def`, so it can `await fetch_page(...)`. Keep your own counter in the loop for the page labels.

</details>

<details><summary><b>Expected output</b></summary>

```text
page 1: ['ada', 'grace']
page 2: ['linus']
```

</details>

<details><summary><b>Solution</b></summary>

```python
class UserPages:
    def __init__(self):
        self.page = 0

    def __aiter__(self):
        return self

    async def __anext__(self):
        self.page += 1
        users = await fetch_page(self.page)
        if not users:
            raise StopAsyncIteration
        return users

number = 0
async for users in UserPages():
    number += 1
    print(f"page {number}: {users}")
```

The async iteration protocol mirrors NB22's: `__aiter__` returns the iterator (a plain method), and `__anext__` is a coroutine returning the next value. `StopAsyncIteration` plays the role of `StopIteration`. Because `__anext__` awaits, each page is fetched only when the loop asks for it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
async def user_pages():
    page = 1
    while users := await fetch_page(page):
        yield page, users
        page += 1

async for number, users in user_pages():
    print(f"page {number}: {users}")
```

An async generator does the same job in five lines: no class, no manual `StopAsyncIteration`. The walrus (NB18) fetches and tests in one step. Write the class only when the iterator needs extra methods or state callers can see.

</details>

In [ ]:
import asyncio

PAGES = {1: ["ada", "grace"], 2: ["linus"], 3: []}   # an empty page means: no more users

async def fetch_page(number):
    await asyncio.sleep(0)
    return PAGES[number]

# your code here


### NB27-Q26 · L4 · gather · Refactor

This prices each item one after the other, so three 0.01 s waits take
0.03 s. Make the lookups run at the same time with `asyncio.gather`,
keeping the output identical.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
import asyncio

MENU = {"tea": 2.5, "cake": 3.0, "juice": 4.25}

async def price_of(item):
    await asyncio.sleep(0.01)
    return MENU[item]

items = ["tea", "cake", "juice"]
prices = []
for item in items:
    price = await price_of(item)
    prices.append(price)
print(prices)
print("total:", sum(prices))
```

<details><summary><b>Hint</b></summary>

Build all three coroutines first, unpack them into `gather` with `*`, and await once. `gather` gives you a list.

</details>

<details><summary><b>Expected output</b></summary>

```text
[2.5, 3.0, 4.25]
total: 9.75
```

</details>

<details><summary><b>Solution</b></summary>

```python
import asyncio

MENU = {"tea": 2.5, "cake": 3.0, "juice": 4.25}

async def price_of(item):
    await asyncio.sleep(0.01)
    return MENU[item]

items = ["tea", "cake", "juice"]
prices = await asyncio.gather(*(price_of(item) for item in items))
print(prices)
print("total:", sum(prices))
```

`await` in a loop runs the coroutines *sequentially*: each one must finish before the next starts. `gather` starts them all and waits once, so the total time is the slowest lookup, not the sum. The results come back as a list in argument order, so the output is unchanged.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import asyncio

MENU = {"tea": 2.5, "cake": 3.0, "juice": 4.25}

async def price_of(item):
    await asyncio.sleep(0.01)
    return MENU[item]

items = ["tea", "cake", "juice"]
async with asyncio.TaskGroup() as tg:
    tasks = [tg.create_task(price_of(item)) for item in items]
prices = [task.result() for task in tasks]
print(prices)
print("total:", sum(prices))
```

`TaskGroup` gives the same speed-up plus safer failure handling: if one lookup raises, the others are cancelled instead of left running. For a thousand items, add a `Semaphore` (Q21) so you don't open a thousand connections at once.

</details>

In [ ]:
import asyncio

MENU = {"tea": 2.5, "cake": 3.0, "juice": 4.25}

async def price_of(item):
    await asyncio.sleep(0.01)
    return MENU[item]

items = ["tea", "cake", "juice"]
prices = []
for item in items:
    price = await price_of(item)
    prices.append(price)
print(prices)
print("total:", sum(prices))


### NB27-Q27 · L4 · TaskGroup · Error

One charge in the group has a bad amount. Which error does this raise?

**Which error does this raise?**

```python
import asyncio

async def charge(card, amount):
    await asyncio.sleep(0)
    if amount <= 0:
        raise ValueError(f"bad amount for {card}")
    return f"{card} charged {amount}"

print("charging cards")
async with asyncio.TaskGroup() as tg:
    tg.create_task(charge("visa", 20))
    tg.create_task(charge("amex", -5))
print("all charged")
```

<details><summary><b>Hint</b></summary>

A task group can have several failing tasks at once. How does it report more than one exception? (NB12)

</details>

<details><summary><b>Expected output</b></summary>

```text
charging cards
ExceptionGroup
```

</details>

<details><summary><b>Solution</b></summary>

`ExceptionGroup`: a `TaskGroup` always wraps task failures in an `ExceptionGroup`, even when only one task failed, because several tasks *could* fail together. The `ValueError` is inside it. A plain `except ValueError:` won't catch it; `except* ValueError:` will.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import asyncio

async def charge(card, amount):
    await asyncio.sleep(0)
    if amount <= 0:
        raise ValueError(f"bad amount for {card}")
    return f"{card} charged {amount}"

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(charge("visa", 20))
        tg.create_task(charge("amex", -5))
except* ValueError as group:
    for error in group.exceptions:
        print("rejected:", error)
```

`except*` (NB12) picks the matching exceptions out of the group. If you'd rather get each task's outcome separately and keep the good ones, use `gather(..., return_exceptions=True)` (Q29).

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## L5 · Hard

### NB27-Q28 · L5 · Rate-limited fetcher · Write

Write `fetch(url, limit)` for a flaky fake server:

- at most 2 requests at once (`limit` is an `asyncio.Semaphore(2)`);
- each attempt gets 0.02 seconds (`asyncio.timeout`);
- on `TimeoutError`, retry, up to 3 attempts in total;
- return the response, or `"<url> gave up"` after 3 timeouts.

Fetch all `urls` with `gather`, print each result, then the total number
of attempts.

```text
/a 200
/b 200
/c 200
/d 200
/e gave up
total attempts: 10
```

**Starting code (already in the cell below):**

```python
import asyncio

attempts = {}
SLOW_ON_TRY = {"/b": [1], "/d": [1, 2], "/e": [1, 2, 3]}   # which tries hang

async def fake_get(url):
    attempts[url] = attempts.get(url, 0) + 1
    if attempts[url] in SLOW_ON_TRY.get(url, []):
        await asyncio.sleep(0.05)       # too slow: the timeout should cut it off
    else:
        await asyncio.sleep(0)
    return f"{url} 200"

urls = ["/a", "/b", "/c", "/d", "/e"]
```

<details><summary><b>Hint</b></summary>

Loop `for attempt in range(3):`. Inside, `async with limit:` then `async with asyncio.timeout(0.02):`, and `return` the response straight away. Catch `TimeoutError` around the timeout block.

</details>

<details><summary><b>Expected output</b></summary>

```text
/a 200
/b 200
/c 200
/d 200
/e gave up
total attempts: 10
```

</details>

<details><summary><b>Solution</b></summary>

```python
async def fetch(url, limit):
    for attempt in range(3):
        try:
            async with limit:
                async with asyncio.timeout(0.02):
                    return await fake_get(url)
        except TimeoutError:
            pass
    return f"{url} gave up"

limit = asyncio.Semaphore(2)
results = await asyncio.gather(*(fetch(url, limit) for url in urls))
for result in results:
    print(result)
print("total attempts:", sum(attempts.values()))
```

Three tools stack up. The semaphore caps concurrency, the timeout caps each attempt, and the loop retries. `return` inside the `async with` blocks leaves both cleanly, releasing the slot. `/a` and `/c` succeed first time (1 + 1), `/b` on its second try (2), `/d` on its third (3), and `/e` hangs on all three tries (3): 10 attempts. The trap is where the semaphore sits: taking the slot *inside* the loop means a URL waiting to retry doesn't hog a slot, and the timeout only starts once the slot is acquired, so time spent queueing isn't counted against an attempt.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
async def fetch(url, limit, *, tries=3, per_try=0.02):
    for attempt in range(1, tries + 1):
        try:
            async with limit, asyncio.timeout(per_try):
                return await fake_get(url)
        except TimeoutError:
            if attempt < tries:
                await asyncio.sleep(0)      # real code: exponential backoff here
    return f"{url} gave up"

limit = asyncio.Semaphore(2)
results = await asyncio.gather(*(fetch(url, limit) for url in urls))
print(*results, sep="\n")
print("total attempts:", sum(attempts.values()))
```

Two async context managers combine in one `async with a, b:`. The retry count and timeout become keyword-only settings (NB09). Real clients wait longer between tries (exponential backoff with jitter, e.g. 0.1 s, 0.2 s, 0.4 s) so a struggling server isn't hit by a wave of instant retries.

</details>

In [ ]:
import asyncio

attempts = {}
SLOW_ON_TRY = {"/b": [1], "/d": [1, 2], "/e": [1, 2, 3]}   # which tries hang

async def fake_get(url):
    attempts[url] = attempts.get(url, 0) + 1
    if attempts[url] in SLOW_ON_TRY.get(url, []):
        await asyncio.sleep(0.05)       # too slow: the timeout should cut it off
    else:
        await asyncio.sleep(0)
    return f"{url} 200"

urls = ["/a", "/b", "/c", "/d", "/e"]

# your code here


### NB27-Q29 · L5 · gather · Fix

One sensor is offline, and its `ConnectionError` crashes the whole
report, losing the two good readings. Fix it so every sensor gets a line,
followed by a summary:

```text
kitchen: ok
garage: offline (ConnectionError)
attic: ok
2 of 3 sensors ok
```

**Buggy code (copied into the cell below):**

```python
import asyncio

async def read_sensor(name):
    await asyncio.sleep(0)
    if name == "garage":
        raise ConnectionError(f"{name} offline")
    return f"{name}: ok"

sensors = ["kitchen", "garage", "attic"]
results = await asyncio.gather(*(read_sensor(name) for name in sensors))
for result in results:
    print(result)
print(len(results), "of", len(sensors), "sensors ok")
```

<details><summary><b>Hint</b></summary>

`gather` has an option that puts each exception into the results list instead of raising it. Then check each result with `isinstance`.

</details>

<details><summary><b>Expected output</b></summary>

```text
kitchen: ok
garage: offline (ConnectionError)
attic: ok
2 of 3 sensors ok
```

</details>

<details><summary><b>Solution</b></summary>

```python
import asyncio

async def read_sensor(name):
    await asyncio.sleep(0)
    if name == "garage":
        raise ConnectionError(f"{name} offline")
    return f"{name}: ok"

sensors = ["kitchen", "garage", "attic"]
results = await asyncio.gather(*(read_sensor(name) for name in sensors),
                               return_exceptions=True)
ok = 0
for name, result in zip(sensors, results):
    if isinstance(result, Exception):
        print(f"{name}: offline ({type(result).__name__})")
    else:
        print(result)
        ok += 1
print(ok, "of", len(sensors), "sensors ok")
```

By default `gather` raises the first exception, and you lose every other result (the other coroutines even keep running in the background). With `return_exceptions=True`, an exception becomes an ordinary item in the results list, in its argument position. `zip` lines each result up with its sensor name. The old summary also counted *all* results as ok; counting has to skip the failures.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import asyncio

async def read_sensor(name):
    await asyncio.sleep(0)
    if name == "garage":
        raise ConnectionError(f"{name} offline")
    return f"{name}: ok"

sensors = ["kitchen", "garage", "attic"]
results = await asyncio.gather(*map(read_sensor, sensors), return_exceptions=True)
failures = {name: err for name, err in zip(sensors, results)
            if isinstance(err, BaseException)}
for name, result in zip(sensors, results):
    print(f"{name}: offline ({type(failures[name]).__name__})" if name in failures
          else result)
print(len(sensors) - len(failures), "of", len(sensors), "sensors ok")
```

`return_exceptions=True` also returns `CancelledError`, which is a `BaseException`, not an `Exception`; seniors test for `BaseException` so a cancelled sensor can't be mistaken for a reading. Never use it to hide errors: log or report every one, as here.

</details>

In [ ]:
import asyncio

async def read_sensor(name):
    await asyncio.sleep(0)
    if name == "garage":
        raise ConnectionError(f"{name} offline")
    return f"{name}: ok"

sensors = ["kitchen", "garage", "attic"]
results = await asyncio.gather(*(read_sensor(name) for name in sensors))
for result in results:
    print(result)
print(len(results), "of", len(sensors), "sensors ok")


### NB27-Q30 · L5 · Worker threads · Write

Three worker threads share the `jobs` queue of invoice amounts. Each worker
takes amounts until it gets `None`, works out a 2% fee
(`round(amount * 0.02, 2)`), and appends `(amount, fee)` to `processed`
while holding `processed_lock`.

Put one `None` per worker on the queue, start the workers, join them, then
print the invoices sorted by amount and the total fee rounded to 2 places.

```text
18 -> fee 0.36
42 -> fee 0.84
75 -> fee 1.5
120 -> fee 2.4
260 -> fee 5.2
300 -> fee 6.0
total fees: 16.3
```

**Starting code (already in the cell below):**

```python
import queue
import threading

jobs = queue.Queue()
for amount in [120, 75, 300, 42, 18, 260]:
    jobs.put(amount)

processed = []
processed_lock = threading.Lock()
```

<details><summary><b>Hint</b></summary>

Put the three `None`s on the queue *after* the real jobs. Sort `processed` (a list of tuples sorts by the first item) before printing, and add the fees up in that sorted order.

</details>

<details><summary><b>Expected output</b></summary>

```text
18 -> fee 0.36
42 -> fee 0.84
75 -> fee 1.5
120 -> fee 2.4
260 -> fee 5.2
300 -> fee 6.0
total fees: 16.3
```

</details>

<details><summary><b>Solution</b></summary>

```python
def worker():
    while True:
        amount = jobs.get()
        if amount is None:
            break
        fee = round(amount * 0.02, 2)
        with processed_lock:
            processed.append((amount, fee))

for _ in range(3):
    jobs.put(None)
workers = [threading.Thread(target=worker) for _ in range(3)]
for w in workers:
    w.start()
for w in workers:
    w.join()

total = 0
for amount, fee in sorted(processed):
    print(amount, "-> fee", fee)
    total += fee
print("total fees:", round(total, 2))
```

Each worker stops at the first `None` it takes, and three `None`s stop three workers. Which worker handles which invoice, and in what order, changes every run; sorting makes the printout stable. The total is summed *after* sorting because float addition depends on order: adding the same floats in a different order can change the last digit.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import math

def worker():
    for amount in iter(jobs.get, None):
        fee = round(amount * 0.02, 2)
        with processed_lock:
            processed.append((amount, fee))

WORKERS = 3
for _ in range(WORKERS):
    jobs.put(None)
workers = [threading.Thread(target=worker, name=f"invoicer-{n}") for n in range(WORKERS)]
for w in workers:
    w.start()
for w in workers:
    w.join()

for amount, fee in sorted(processed):
    print(amount, "-> fee", fee)
print("total fees:", round(math.fsum(fee for _, fee in processed), 2))
```

`math.fsum` adds floats exactly, so the total no longer depends on order. A named constant ties the number of sentinels to the number of workers. For money, real code would store cents as `int` or use `Decimal` (NB16). `list.append` is atomic in CPython, but the explicit lock documents intent and stays correct on other implementations.

</details>

In [ ]:
import queue
import threading

jobs = queue.Queue()
for amount in [120, 75, 300, 42, 18, 260]:
    jobs.put(amount)

processed = []
processed_lock = threading.Lock()

# your code here


### NB27-Q31 · L5 · Timeouts and cleanup · Predict

An upload takes longer than its timeout allows. The upload has a
`finally` block. Predict the log.

**What does this print?**

```python
import asyncio

log = []

async def upload():
    try:
        log.append("upload start")
        await asyncio.sleep(0.05)
        log.append("upload done")
    finally:
        log.append("cleanup")

try:
    async with asyncio.timeout(0.01):
        await upload()
    log.append("after upload")
except TimeoutError:
    log.append("timed out")
print(*log, sep="\n")
```

<details><summary><b>Hint</b></summary>

A timeout works by *cancelling* the task: a `CancelledError` appears at the `await` that was waiting. What runs on the way out?

</details>

<details><summary><b>Expected output</b></summary>

```text
upload start
cleanup
timed out
```

</details>

<details><summary><b>Solution</b></summary>

At 0.01 s the timeout cancels the waiting code: a `CancelledError` is raised at `await asyncio.sleep(0.05)`. So "upload done" never happens, but the `finally` runs ("cleanup"). As the error leaves `asyncio.timeout`, it becomes `TimeoutError`, skipping "after upload" and landing in the `except`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Cancellation is just an exception, so `finally` and `async with` cleanup always run, which is why you release resources there. Never swallow `CancelledError` with a bare `except:` or `except BaseException:`; if you catch it to clean up, re-raise it, or timeouts and shutdowns stop working.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB27-Q32 · L5 · as_completed · Write

Resize photos in a pool of 3 threads. `resize` raises `ValueError` for
GIF files. Submit every photo, collect successes and failures with
`as_completed`, then print the successes sorted, then the failures
sorted, with the exception's class name.

```text
ok: beach.jpg resized
ok: cat.png resized
ok: dog.jpg resized
failed: logo.gif (ValueError)
failed: party.gif (ValueError)
```

**Starting code (already in the cell below):**

```python
from concurrent.futures import ThreadPoolExecutor, as_completed

def resize(photo):
    if photo.endswith(".gif"):
        raise ValueError(f"cannot resize {photo}")
    return f"{photo} resized"

photos = ["beach.jpg", "party.gif", "cat.png", "logo.gif", "dog.jpg"]
```

<details><summary><b>Hint</b></summary>

Keep a dict from each future to its photo, so a failed future can still tell you its file name. A worker's exception is raised again when you call `future.result()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
ok: beach.jpg resized
ok: cat.png resized
ok: dog.jpg resized
failed: logo.gif (ValueError)
failed: party.gif (ValueError)
```

</details>

<details><summary><b>Solution</b></summary>

```python
successes = []
failures = []
with ThreadPoolExecutor(max_workers=3) as pool:
    future_to_photo = {}
    for photo in photos:
        future_to_photo[pool.submit(resize, photo)] = photo
    for future in as_completed(future_to_photo):
        photo = future_to_photo[future]
        try:
            successes.append(future.result())
        except ValueError as error:
            failures.append(f"{photo} ({type(error).__name__})")
for message in sorted(successes):
    print("ok:", message)
for message in sorted(failures):
    print("failed:", message)
```

An exception inside a worker thread doesn't crash your program: the future stores it, and `future.result()` raises it again in *your* thread. If you never call `result()`, the error vanishes silently, which is the trap. The dict remembers which photo each future belongs to, and sorting fixes the random finishing order.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
successes, failures = [], []
with ThreadPoolExecutor(max_workers=3) as pool:
    future_to_photo = {pool.submit(resize, photo): photo for photo in photos}
    for future in as_completed(future_to_photo):
        if (error := future.exception()) is not None:
            failures.append(f"{future_to_photo[future]} ({type(error).__name__})")
        else:
            successes.append(future.result())
print(*(f"ok: {m}" for m in sorted(successes)), sep="\n")
print(*(f"failed: {m}" for m in sorted(failures)), sep="\n")
```

`future.exception()` returns the error (or `None`) without raising, which avoids a `try` and catches *any* exception type, not just the one you expected. A dict comprehension (NB10) builds the future-to-photo map in one line.

</details>

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def resize(photo):
    if photo.endswith(".gif"):
        raise ValueError(f"cannot resize {photo}")
    return f"{photo} resized"

photos = ["beach.jpg", "party.gif", "cat.png", "logo.gif", "dog.jpg"]

# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB27-P · L5 · Mini-project: Async Site Crawler · Write

Crawl a fake website, starting at `/`, level by level (breadth-first).

- Fetch every page of one level at the same time with `gather`, but never
  more than 3 at once (a `Semaphore(3)`), and record the peak number of
  fetches running at once.
- Never fetch a page twice.
- `fetch_links` raises `LookupError` for a page that doesn't exist; mark
  that URL as broken and keep going.
- For each level, print its URLs sorted. At the end print the broken URLs
  and a summary.

```text
depth 0: /
depth 1: /about /blog /shop
depth 2: /blog/1 /blog/2 /old-post /shop/cart /team
broken: /old-post
fetched 9 pages, 1 broken, peak 3 at once
```

**Starting code (already in the cell below):**

```python
import asyncio

SITE = {
    "/": ["/about", "/blog", "/shop"],
    "/about": ["/", "/team"],
    "/blog": ["/blog/1", "/blog/2", "/old-post"],
    "/shop": ["/shop/cart", "/"],
    "/team": [],
    "/blog/1": ["/blog/2"],
    "/blog/2": ["/blog/1", "/about"],
    "/shop/cart": ["/shop"],
}

async def fetch_links(url):
    """Pretend to download a page and return the links on it."""
    await asyncio.sleep(0.01)
    if url not in SITE:
        raise LookupError(f"404 {url}")
    return SITE[url]
```

<details><summary><b>Hint</b></summary>

Write `visit(url)`: inside `async with limit:`, count `active` up, update `peak`, fetch, and count down in a `finally`. Return `None` for a broken page. The main loop keeps a `seen` set and a list for the current level; each round gathers the level and builds the next one from links not yet seen.

</details>

<details><summary><b>Expected output</b></summary>

```text
depth 0: /
depth 1: /about /blog /shop
depth 2: /blog/1 /blog/2 /old-post /shop/cart /team
broken: /old-post
fetched 9 pages, 1 broken, peak 3 at once
```

</details>

<details><summary><b>Solution</b></summary>

```python
limit = asyncio.Semaphore(3)
active = 0
peak = 0

async def visit(url):
    global active, peak
    async with limit:
        active += 1
        peak = max(peak, active)
        try:
            return await fetch_links(url)
        except LookupError:
            return None
        finally:
            active -= 1

seen = {"/"}
level = ["/"]
broken = []
fetched = 0
depth = 0
while level:
    print(f"depth {depth}:", " ".join(sorted(level)))
    results = await asyncio.gather(*(visit(url) for url in level))
    fetched += len(level)
    next_level = []
    for url, links in zip(level, results):
        if links is None:
            broken.append(url)
            continue
        for link in links:
            if link not in seen:
                seen.add(link)
                next_level.append(link)
    level = next_level
    depth += 1

print("broken:", " ".join(sorted(broken)))
print(f"fetched {fetched} pages, {len(broken)} broken, peak {peak} at once")
```

Breadth-first crawling handles one level at a time: `gather` fetches the whole level concurrently, and the semaphore keeps it to 3 at once (the five pages of depth 2 hit that limit). Adding a link to `seen` the moment it's *discovered*, not when it's fetched, stops two pages from both queueing the same link. Catching `LookupError` inside `visit` turns a broken page into a `None` result, so one 404 can't cancel the crawl. No lock is needed for `active`, `peak` or `seen`: asyncio only switches tasks at `await`, and none of those updates contain one.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass, field

@dataclass
class Crawler:
    max_parallel: int = 3
    active: int = 0
    peak: int = 0
    fetched: int = 0
    broken: set = field(default_factory=set)

    def __post_init__(self):
        self._limit = asyncio.Semaphore(self.max_parallel)

    async def visit(self, url):
        async with self._limit:
            self.active += 1
            self.peak = max(self.peak, self.active)
            self.fetched += 1
            try:
                return await fetch_links(url)
            except LookupError:
                self.broken.add(url)
                return []
            finally:
                self.active -= 1

    async def crawl(self, start):
        seen, level, depth = {start}, [start], 0
        while level:
            print(f"depth {depth}:", " ".join(sorted(level)))
            async with asyncio.TaskGroup() as tg:
                tasks = [tg.create_task(self.visit(url)) for url in level]
            level = []
            for task in tasks:
                for link in task.result():
                    if link not in seen:
                        seen.add(link)
                        level.append(link)
            depth += 1

crawler = Crawler()
await crawler.crawl("/")
print("broken:", " ".join(sorted(crawler.broken)))
print(f"fetched {crawler.fetched} pages, {len(crawler.broken)} broken, "
      f"peak {crawler.peak} at once")
```

A small class (a dataclass, NB21) keeps the crawler's state together instead of in globals, so you can run two crawls side by side. `TaskGroup` replaces `gather` for structured concurrency. A production crawler would also respect `robots.txt`, add per-request timeouts (Q15) and retries (Q28), and use an `asyncio.Queue` of worker tasks instead of strict levels, so one slow page doesn't hold up the next level.

</details>

In [ ]:
import asyncio

SITE = {
    "/": ["/about", "/blog", "/shop"],
    "/about": ["/", "/team"],
    "/blog": ["/blog/1", "/blog/2", "/old-post"],
    "/shop": ["/shop/cart", "/"],
    "/team": [],
    "/blog/1": ["/blog/2"],
    "/blog/2": ["/blog/1", "/about"],
    "/shop/cart": ["/shop"],
}

async def fetch_links(url):
    """Pretend to download a page and return the links on it."""
    await asyncio.sleep(0.01)
    if url not in SITE:
        raise LookupError(f"404 {url}")
    return SITE[url]

# your code here


---

## Done

Next up: **NB28 · Testing Debugging Logging** in `advanced/`.